# Variant mining from recipe data

Sarthak Aggarwal (MT26139) and Divyanshu Joshi (MT26120), CGAS Group 26, Deliverable 1

**Question.** Our `roman_variants` column was collected by hand and with AI help. Do real recipes actually use those spellings? And which common spellings did we miss?

**Method.**
1. Read every ingredient phrase from the recipe file (10,000 recipes).
2. Build a lookup: every English name and every Roman variant points to its ingredient (a dictionary, so one lookup per phrase).
3. **Coverage.** Count how many of our ingredients appear in the recipes, and which of our variants are really used.
4. **Missed spellings.** Take the frequent recipe phrases that match nothing in our list, and suggest the closest ingredient by shared words. These are candidates only; a person decides whether to add them.

Files needed in the same folder: `D1_0_ingredient_250_final_dataset.csv` and `D1_input_recipes_10000.csv`.

In [ ]:
import csv
import collections
import re
import pandas as pd

RECIPES = "D1_input_recipes_10000.csv"
DATASET = "D1_0_ingredient_250_final_dataset.csv"

recipes = []
with open(RECIPES, encoding="utf-8-sig", newline="") as f:
    for row in csv.DictReader(f):
        text = row["Ingredients_List"].strip().strip("[]")
        recipes.append([p.strip().lower() for p in text.split(",") if p.strip()])

phrase_count = collections.Counter(p for r in recipes for p in set(r))   # recipes containing each phrase
print(len(recipes), "recipes |", len(phrase_count), "different ingredient phrases")
phrase_count.most_common(10)

## Step 1. Lookup table
Every name and variant maps to its ingredient. If a spelling were on two ingredients we would keep the first, but `D1_4_validate.py` already makes sure that does not happen.

In [ ]:
dataset = pd.read_csv(DATASET, encoding="utf-8-sig")

lookup = {}        # spelling -> (ingredient, kind)
for _, d in dataset.iterrows():
    lookup.setdefault(d["english_name"], (d["english_name"], "english name"))
    for v in d["roman_variants"].split("|"):
        lookup.setdefault(v, (d["english_name"], "variant"))
print(len(lookup), "spellings in total")

## Step 2. Coverage
How many recipes use each ingredient, counted by exact match of the whole phrase.

In [ ]:
used = collections.Counter()
used_by_kind = collections.Counter()
spelling_hits = {}
for phrase, n in phrase_count.items():
    if phrase in lookup:
        name, kind = lookup[phrase]
        used[name] += n
        used_by_kind[kind] += n
        spelling_hits[phrase] = n

total = len(dataset)
print(f"{len(used)} of {total} ingredients appear in at least one recipe ({len(used) / total:.0%})")
print("recipe phrases matched through the English name:", used_by_kind["english name"])
print("recipe phrases matched through a Roman variant: ", used_by_kind["variant"])
pd.DataFrame(used.most_common(15), columns=["ingredient", "recipes using it"])

In [ ]:
# variants that real recipes use (these are the ones that prove the column is useful)
variant_hits = [(p, lookup[p][0], n) for p, n in spelling_hits.items() if lookup[p][1] == "variant"]
variant_hits.sort(key=lambda t: -t[2])
pd.DataFrame(variant_hits[:20], columns=["spelling in recipes", "our ingredient", "recipes"])

In [ ]:
never_used = sorted(set(dataset["english_name"]) - set(used))
print(len(never_used), "ingredients never matched:")
print(", ".join(never_used))

## Step 3. Spellings we may have missed
Take recipe phrases that are not in our lookup and appear in at least `MIN_RECIPES` recipes. For each one, find the ingredient that shares the most words (word overlap divided by total words, the Jaccard score). Phrases with no shared word are skipped.

This is a suggestion list, not an automatic fix: "ginger" could mean fresh ginger, ginger paste or dry ginger powder, so a person has to choose.

In [ ]:
MIN_RECIPES = 40

def words(text):
    out = set()
    for w in re.findall(r"[a-z]+", text):
        out.add(w[:-1] if w.endswith("s") and len(w) > 3 else w)
    return out

candidates = [(name, words(sp)) for sp, (name, _) in lookup.items()]

rows = []
for phrase, n in phrase_count.most_common():
    if n < MIN_RECIPES:
        break
    if phrase in lookup:
        continue
    pw = words(phrase)
    best, score = None, 0.0
    for name, cw in candidates:
        overlap = len(pw & cw)
        if overlap:
            s = overlap / len(pw | cw)
            if s > score:
                best, score = name, s
    rows.append({"recipe phrase": phrase, "recipes": n, "closest ingredient": best or "", "score": round(score, 2)})

missed = pd.DataFrame(rows)
print(len(missed), "frequent phrases are not in our list")
missed.head(40)

## Step 4. Save the findings
The table is saved so the report can quote it.

In [ ]:
missed.to_csv("D1_6_variant_mining_missed.csv", index=False, encoding="utf-8-sig")
print("saved D1_6_variant_mining_missed.csv with", len(missed), "rows")

## What to say about the result
* Most of our ingredients do appear in the recipes, so the names and variants are used in real data.
* The recipe file already uses many of our English names and several variants (for example `eggplant`, `paneer`, `zucchini`), so the variant column matches real usage, not only AI guesses.
* The missed list shows common shorter or alternative phrases (for example plain `turmeric`, `ginger`, `green chilli`) that point to one of our rows. These are the first candidates to add as variants.
* Limits: matching is on whole phrases, so spellings with extra words (`fresh cilantro leaves`) are only found through the word-overlap suggestions. The recipe file is mostly English, so it can confirm English synonyms better than Hindi transliterations.